(DNA) Make an artificial sequence of 200 up tp 400 bases in length in which to simulate 3-4 transposable elements. Implement a software application to detect these transposable elements (star, end) within the created DNA sequence.

it just needs to find the inverted repeats

In between those patterns there's usually a gene which is a file. when executed it will generate an rmna molecule. from it a protein will emerge. which is an enzyme. it'll come back, it reocignises the repeats, it cuts, it replaces, it starts floating ???

take a bacterial genome fasta file, make it the input, look for NEW possible transposones. (we do not know the inverted )

hint: take sliding windows into consideration

In [2]:
import random
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import HTML, display

COMPLEMENT = str.maketrans("ACGT", "TGCA")

def reverse_complement(seq: str) -> str:
    # Reverse complement of a DNA string.
    return seq.translate(COMPLEMENT)[::-1]

print(reverse_complement("GATCCTAGGCAT"))   # -> ATGCCTAGGATC

ATGCCTAGGATC


In [3]:
import random
import pandas as pd

random.seed(7)

NUM_TRANSP = 4          # 3 or 4 transposable elements
SEQ_LEN = 300        # total length (200-400 bp)
TIR_LEN = 12         # length of each inverted repeat

def random_dna(n):
    return "".join(random.choice("ACGT") for _ in range(n))

# 1. TIR + random middle + reverse complement of TIR
TRANSP = []
for _ in range(NUM_TRANSP):
    tir = random_dna(TIR_LEN)
    middle = random_dna(random.randint(20, 35))
    TRANSP.append((tir, tir + middle + reverse_complement(tir)))

# 2. random spacer lengths (at least 5 pairs each) that fill up the rest of the sequence
free = SEQ_LEN - sum(len(te) for _, te in TRANSP) - 5 * (NUM_TRANSP + 1)
cuts = sorted(random.randint(0, free) for _ in range(NUM_TRANSP))
spacers = [5 + b - a for a, b in zip([0] + cuts, cuts + [free])]

# 3. assemble the sequence and record where each element was placed
seq, truth = "", []
for i, (tir, te) in enumerate(TRANSP):
    seq += random_dna(spacers[i])
    truth.append({"TE": f"TE{i+1}", "start": len(seq) + 1, "end": len(seq) + len(te),
                  "length": len(te), "TIR_len": TIR_LEN,
                  "left_TIR": tir, "right_TIR": reverse_complement(tir)})
    seq += te
seq += random_dna(spacers[-1])

# 4. make sure the bases just outside each element don't pair by chance
#    (otherwise the inverted repeat would really be 1 bp longer than planned)
seq = list(seq)
for t in truth:
    before, after = t["start"] - 2, t["end"]
    while seq[before] == reverse_complement(seq[after]):
        seq[before] = random.choice("ACGT")
sequence = "".join(seq)
truth_df = pd.DataFrame(truth)

print(f"Sequence length: {len(sequence)} bp\n")
for i in range(0, len(sequence), 60):
    print(f"{i+1:>4}  {sequence[i:i+60]}")
print("\nPlanted transposable elements:")
display(truth_df)

Sequence length: 300 bp

   1  AGGTCACGCAGAGGTGCTAAAGACAATACATAACATACACGTCAGCACGAAACTTGTTGG
  61  ATTGTCTTTAGCGCGCCCTCCTGAACCCAGTGTGAATGCTTAAGGGTTAAGTAAGTGTGA
 121  TGATTCACACTGGGGTGCGTGGACACTCGCTATGAATCTCCATACGCCTTTATTGCTGTG
 181  TCCACCCCATCGGACTGTAAAGGCGTATGTGATTTACCCACTCTGCCAAACTCCACGCAT
 241  TTTTATTAACTCAGAAACAGAACTCGGGTAATTTTAATAAAAATGCCGCGGTCAGTTCCA

Planted transposable elements:


,TE,start,end,length,TIR_len,left_TIR,right_TIR
0,TE1,16,72,57,12,GCTAAAGACAAT,ATTGTCTTTAGC
1,TE2,86,134,49,12,CCCAGTGTGAAT,ATTCACACTGGG
2,TE3,161,209,49,12,CATACGCCTTTA,TAAAGGCGTATG
3,TE4,237,286,50,12,GCATTTTTATTA,TAATAAAAATGC


In [4]:
def find_inverted_repeats(seq, min_tir=10, min_internal=10, max_te_len=200):
    # Return a list of terminal inverted repeats (candidate transposable elements).
    # Coordinates are returned 1-based and inclusive.
    seq = seq.upper()
    n, k = len(seq), min_tir

    # 1. k-mer index
    index = {}
    for p in range(n - k + 1):
        index.setdefault(seq[p:p + k], []).append(p)

    comp = {"A": "T", "T": "A", "G": "C", "C": "G"}
    found = {}

    for i in range(n - k + 1):
        target = reverse_complement(seq[i:i + k])
        for j in index.get(target, []):
            # right TIR must be downstream with an internal region in between
            if j < i + k + min_internal or (j + k - i) > max_te_len:
                continue

            # 3a. extend inward: seq[i+k+t] pairs with seq[j-1-t]
            inward = 0
            while (i + k + inward) < (j - 1 - inward) - min_internal and \
                  comp.get(seq[i + k + inward]) == seq[j - 1 - inward]:
                inward += 1
            # 3b. extend outward: seq[i-1-t] pairs with seq[j+k+t]
            outward = 0
            while i - 1 - outward >= 0 and j + k + outward < n and \
                  comp.get(seq[i - 1 - outward]) == seq[j + k + outward]:
                outward += 1

            start = i - outward
            end = j + k + outward
            tir_len = k + inward + outward
            # 4. de-duplicate (keep the longest TIR for each start/end)
            key = (start, end)
            if key not in found or found[key] < tir_len:
                found[key] = tir_len

    results = []
    for (start, end), tir_len in sorted(found.items()):
        results.append({"start": start + 1, "end": end, "length": end - start, "TIR_len": tir_len,
                        "left_TIR": seq[start:start + tir_len],
                        "right_TIR": seq[end - tir_len:end]})
    return results

detected = find_inverted_repeats(sequence)
detected_df = pd.DataFrame(detected)
detected_df.insert(0, "hit", [f"hit{i+1}" for i in range(len(detected_df))])
print(f"Detected {len(detected_df)} inverted-repeat element(s):")
display(detected_df)

# check that each pair really is an inverted repeat
for r in detected:
    assert reverse_complement(r["left_TIR"]) == r["right_TIR"]

Detected 4 inverted-repeat element(s):


,hit,start,end,length,TIR_len,left_TIR,right_TIR
0,hit1,16,72,57,12,GCTAAAGACAAT,ATTGTCTTTAGC
1,hit2,86,134,49,12,CCCAGTGTGAAT,ATTCACACTGGG
2,hit3,161,209,49,12,CATACGCCTTTA,TAAAGGCGTATG
3,hit4,237,286,50,13,GCATTTTTATTAA,TTAATAAAAATGC
